# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RodionOm/Search-ranking-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    !git clone -q https://github.com/flyrank-bih/flyrank-ml-internship-starter.git
    %cd flyrank-ml-internship-starter
print(os.getcwd())

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
y = (df["trend_direction"] == "down").astype(int)
groups = df["client_id"]                      # только для разбиения, НЕ признак

X = pd.DataFrame(index=df.index)

# трафик ДО окна метки: older = 90d - last - prev ; prev = как есть
for m in ["impressions", "clicks", "sessions"]:
    older = df[f"{m}_90d"] - df[f"{m}_last_30d"] - df[f"{m}_prev_30d"]
    X[f"log_{m}_older_30d"] = np.log1p(older)
    X[f"log_{m}_prev_30d"]  = np.log1p(df[f"{m}_prev_30d"])

# импульс до окна метки: prev против older
X["early_momentum"] = X["log_impressions_prev_30d"] - X["log_impressions_older_30d"]

# CTR только в prev-окне (в %, как колонка ctr)
X["ctr_prev_30d"] = np.where(df["impressions_prev_30d"] > 0,
                             df["clicks_prev_30d"] / df["impressions_prev_30d"] * 100, 0)

X["content_age_days"] = df["content_age_days"]

# размер текста: пропуски систематические -> флаг + 0
X["has_word_count"] = df["word_count"].notna().astype(int)
X["word_count"]     = df["word_count"].fillna(0)

# keyword-данные: пропуски по content_type -> флаг + 0
X["has_keyword_data"]  = df["search_volume"].notna().astype(int)
X["log_search_volume"] = np.log1p(df["search_volume"].fillna(0))
X["competition"]       = df["competition"].fillna(0)
X["cpc"]               = df["cpc"].fillna(0)

# категории -> one-hot
cats = df[["content_type", "main_intent"]].fillna("unknown")
X = pd.concat([X, pd.get_dummies(cats, dtype=int)], axis=1)

print(X.shape)                                # (30000, 23)
print("NaN left:", int(X.isna().sum().sum())) # 0

/content/flyrank-ml-internship-starter
/content/flyrank-ml-internship-starter
(30000, 23)
NaN left: 0


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing values | Known before the moment of prediction? |
|---|---|---|---|
| `log_impressions/clicks/sessions_older_30d` | traffic in days 61–90 back | none (derived, checked ≥ 0) | yes |
| `log_impressions/clicks/sessions_prev_30d` | traffic in days 31–60 back | none | yes |
| `early_momentum` | prev vs older impressions (log difference) | none | yes |
| `ctr_prev_30d` | CTR in the prev window, ×100 | 0 when prev impressions = 0 | yes |
| `content_age_days` | days since the page was created | none | yes (shifted by the same 30 days for every page) |
| `word_count` + `has_word_count` | article length | flag + 0 (28% of keyword articles unmeasured) | mostly — measured at export, so a refresh inside the label window could change it (directional caveat) |
| `log_search_volume`, `competition`, `cpc` + `has_keyword_data` | target-keyword context | flag + 0 (feedly articles never have it) | yes (content metadata) |
| `content_type_*`, `main_intent_*` | one-hot article type and intent | blank → `unknown` column | yes |

Missingness is systematic: feedly articles never have keyword data. I add `has_` flags before filling, so the model sees "missing" as an explicit signal instead of a hidden zero. The check below backs this up.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# share of missing values by content type
print(df.groupby("content_type")[["search_volume", "word_count"]]
        .apply(lambda g: g.isna().mean().round(3)))

# share declining with vs without keyword data
print(y.groupby(X["has_keyword_data"]).mean().round(3))

                    search_volume  word_count
content_type                                 
comparison article          0.000       0.000
feedly article              1.000       0.000
keyword article             0.014       0.283
has_keyword_data
0    0.299
1    0.564
Name: trend_direction, dtype: float64


## 3. The leakage hunt

**Method (attack my own features):** train the same model on the honest features, then add one suspect group at a time. If AUC jumps toward 1.0, that suspect is leaking the answer.

- Model: `HistGradientBoostingClassifier` (a tool for the attack here, not the final model).
- Validation: `GroupKFold(5)` by `client_id` — every test fold contains only clients the model never saw.
- Metric: ROC AUC (0.50 = random ranking, 1.00 = perfect ordering). Base rate printed alongside.

**Suspects:**
1. **Label-window column:** `impressions_last_30d`.
2. **Overlapping 90-day window:** `impressions_90d`, `clicks_90d`, `sessions_90d`, `ctr`, `avg_position` — they contain the last 30 days, and together with prev/older they can rebuild it by subtraction.
3. **Future event:** `days_since_last_update` — for most pages the last update happened inside the label window.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, KFold, cross_val_score

def grouped_auc(features):
    model = HistGradientBoostingClassifier(random_state=42)
    scores = cross_val_score(model, features, y, groups=groups,
                             cv=GroupKFold(n_splits=5), scoring="roc_auc")
    return scores.mean()

print(f"base rate: {y.mean():.3f}   (random ranking -> AUC 0.50)\n")

tests = {
    "honest features":                       X,
    "+ impressions_last_30d (label window)": X.assign(imp_last=df["impressions_last_30d"]),
    "+ 90d totals & rates (overlap window)": X.assign(imp90=df["impressions_90d"], clk90=df["clicks_90d"],
                                                      ses90=df["sessions_90d"], ctr=df["ctr"],
                                                      pos=df["avg_position"]),
    "+ days_since_last_update":              X.assign(dslu=df["days_since_last_update"]),
    "+ trend_pct (label source)":            X.assign(trend=df["trend_pct"].fillna(0)),
}
for name, feats in tests.items():
    print(f"{name:40s} AUC {grouped_auc(feats):.3f}")

base rate: 0.542   (random ranking -> AUC 0.50)

honest features                          AUC 0.680
+ impressions_last_30d (label window)    AUC 0.999
+ 90d totals & rates (overlap window)    AUC 0.909
+ days_since_last_update                 AUC 0.700
+ trend_pct (label source)               AUC 1.000


## 4. What I excluded and why

**Second attack: random split vs client split.** A random split puts pages of the same client in both train and test, so the model can memorise the client. The gap between the two numbers measures that memorisation.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
model = HistGradientBoostingClassifier(random_state=42)
rand = cross_val_score(model, X, y, cv=KFold(5, shuffle=True, random_state=42), scoring="roc_auc").mean()
grp  = cross_val_score(model, X, y, groups=groups, cv=GroupKFold(5), scoring="roc_auc").mean()
print(f"random split  AUC {rand:.3f}")
print(f"client split  AUC {grp:.3f}")
print(f"gap           {rand - grp:.3f}")

# proof of the 90d overlap: last 30d can be rebuilt exactly from legal + 90d columns
rebuilt = df["impressions_90d"] - np.expm1(X["log_impressions_prev_30d"]) - np.expm1(X["log_impressions_older_30d"])
print("last_30d rebuilt exactly:", np.allclose(rebuilt, df["impressions_last_30d"]))
print(f"pages updated inside the label window: {(df['days_since_last_update'] < 30).mean():.1%}")

random split  AUC 0.792
client split  AUC 0.680
gap           0.112
last_30d rebuilt exactly: True
pages updated inside the label window: 68.3%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.